<a href="https://colab.research.google.com/github/tamaravdd/iese-dsmba/blob/main/notebooks/02_Python_NumPy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Python & NumPy for Data Analysis

**Session objectives**
1. Learn the Python building blocks you'll use throughout the course
2. Understand NumPy arrays and why they matter for data analysis
3. Apply these tools to a real Airbnb dataset

> **Note on AI tools**: You are encouraged to use ChatGPT / Claude to help with code. The rule is simple: *you must be able to explain every line*. We'll discuss good prompting habits at the end of the notebook.

## Setup

We import NumPy right at the top — it will be our main tool today.

In [ ]:
import numpy as np

---
## 1. Python Essentials

### 1.1 Variables and types

Python infers the **type** of a variable automatically. The types you'll encounter most are `int`, `float`, `bool`, and `str`. Use `type()` to check.

In [ ]:
revenue     = 4_200_000   # int   — annual revenue in €
margin      = 0.32        # float — gross margin
is_listed   = True        # bool  — publicly listed company
company     = 'Inditex'   # str

print(type(revenue), type(margin), type(is_listed), type(company))

Reassigning a variable replaces its previous value:

In [ ]:
margin = margin * 1.05   # margins improved by 5%
margin

**Booleans** arise naturally from comparisons, and behave as `1` / `0` in arithmetic — useful for counting:

In [ ]:
profitable = margin > 0.30
print(profitable)          # True
print(int(profitable))     # 1

### 1.2 Packages

Python ships with a small core. Extra functionality comes from **packages** that you import at the start of a notebook. For example, `math` provides mathematical functions:

In [ ]:
import math
math.log(revenue)    # natural log of annual revenue

Most packages used in this course (NumPy, pandas, matplotlib) are pre-installed in Anaconda and Google Colab. If you ever need something extra, run `pip install packagename` in a terminal, or `!pip install packagename` inside a notebook cell.

### 1.3 Data containers

#### Lists

A **list** holds an ordered sequence of items (any mix of types). You'll mainly use lists as a stepping stone before loading data into NumPy or pandas.

In [ ]:
# Quarterly revenues for a retailer (€ millions)
quarterly = [8.2, 9.5, 11.3, 14.7]

print(quarterly[0])     # first element — Python indexes from 0
print(quarterly[-1])    # last element
print(quarterly[1:3])   # slice: Q2 and Q3 (index 3 is excluded)

Lists feel intuitive, but watch what happens when you try to do maths on them:

In [ ]:
arr = np.array(quarterly)

print('List * 2 :', quarterly * 2)   # repeats the list — not what you want!
print('Array * 2:', arr * 2)         # doubles every value
print('YoY growth:', arr * 1.08)     # apply 8% growth to all quarters

#### Dictionaries

A **dictionary** maps keys to values — a natural fit for a single record (customer, product, transaction):

In [ ]:
customer = {
    'id': 'C-8821',
    'segment': 'Premium',
    'lifetime_value': 4320,
    'purchases': 17,
    'active': True
}

print(customer['segment'])   # access by key
print(customer.keys())

Dictionaries also appear whenever you call a web API. Here's a live example — fetching Bitcoin's current price from CoinGecko:

In [ ]:
import requests

url = 'https://api.coingecko.com/api/v3/coins/markets?vs_currency=usd&ids=bitcoin&order=market_cap_desc&per_page=1&page=1&sparkline=false'
data = requests.get(url).json()[0]

print('Price :', data['current_price'])
print('24h % :', data['price_change_percentage_24h'])

> **Coming up**: When you load a CSV with pandas, each row becomes something very similar to a dictionary. Understanding dicts now makes the transition natural.

### 1.4 Functions

A **function** packages reusable logic. Note the indentation — Python uses it instead of braces.

In [ ]:
def gross_profit(revenue, cost):
    """Returns gross profit and margin."""
    profit = revenue - cost
    margin = profit / revenue
    return profit, margin   # functions can return multiple values

p, m = gross_profit(500_000, 320_000)
print(f'Profit: €{p:,}  |  Margin: {m:.1%}')

Short, single-expression functions can be written as **lambda functions**:

In [ ]:
cagr = lambda start, end, years: (end / start) ** (1 / years) - 1
print(f'CAGR: {cagr(100, 160, 5):.1%}')   # 60% total growth over 5 years

### 1.5 Loops and list comprehensions

In data analysis you'll use loops mainly to build lists before converting them to arrays. The compact form is a **list comprehension**:

In [ ]:
regions = ['North', 'South', 'North', 'West', 'South']

# Flag Northern region entries
is_north = [1 if r == 'North' else 0 for r in regions]
print(is_north)

In practice, NumPy and pandas provide faster, cleaner alternatives to explicit loops — you'll rarely write `for` loops once you know them.

### ✏️ You try it — Airbnb listing profile

a) Create a `dict` for an Airbnb listing with at least 5 attributes (e.g. `neighbourhood`, `room_type`, `price`, `number_of_reviews`, `is_superhost`).  
b) Write a function `host_score(reviews, is_superhost)` that returns `reviews * 1.5` if the host is a superhost, else `reviews * 1.0`.  
c) Call it with the values from your dict.

In [ ]:
# your code here

---
## 2. NumPy Arrays

NumPy is the foundation of almost all numerical work in Python. Its key object is the **ndarray** — a fast, typed, multi-dimensional array.

### 2.1 Arrays vs lists

In [ ]:
sales_list  = [23, 41, 18, 57, 34]   # a plain Python list
sales_array = np.array(sales_list)   # a NumPy array

print(type(sales_list))
print(type(sales_array))
print(sales_array.dtype)

Arrays require all elements to have the same type. If you mix types, NumPy converts everything to the most general type — usually strings, which makes arithmetic impossible:

In [ ]:
mixed = np.array([1, 'North', 3])
mixed   # everything becomes a string

### 2.2 2D arrays and shape

A 2D array is a matrix — rows are observations, columns are variables. Here each row is a sales representative and the columns are `[calls_made, deals_closed, revenue_k€]`:

In [ ]:
# rows = sales reps, cols = [calls_made, deals_closed, revenue (k€)]
sales_reps = np.array([
    [120,  8, 42],
    [ 95, 11, 61],
    [140,  6, 28],
    [ 80, 14, 74],
    [110,  9, 48],
    [130,  7, 35],
])

print('shape:', sales_reps.shape)   # (rows, columns)

`shape` returns a **tuple** — a fixed, immutable sequence. You'll see it everywhere once you start working with data.

In [ ]:
n_reps, n_features = sales_reps.shape
print(f'{n_reps} sales reps, {n_features} features')

### 2.3 Summary statistics

Use `axis=0` to compute across rows (one result per column), `axis=1` to compute across columns (one result per row):

In [ ]:
# Average calls, deals, and revenue across all reps
print('Column means:', np.mean(sales_reps, axis=0))

# Best performer in each category
print('Column maxima:', np.max(sales_reps, axis=0))

In [ ]:
# Focus on revenue column (index 2)
revenue = sales_reps[:, 2]

print('mean   :', np.mean(revenue))
print('median :', np.median(revenue))
print('std    :', np.std(revenue))
print('25th / 75th pct:', np.percentile(revenue, 25), '/', np.percentile(revenue, 75))

### 2.4 Vectorized functions

Functions written with NumPy operations work element-wise on entire arrays — no loops needed. Here is the classic example: BMI from a population health dataset.

In [ ]:
# rows = employees at a corporate wellness check, cols = [height_cm, weight_kg]
employees = np.array([
    [160, 55], [175, 70], [155, 52], [180, 77],
    [165, 58], [170, 68], [162, 54], [178, 75],
    [158, 53], [172, 72]
])

def calc_bmi(h_cm, w_kg):
    return w_kg / (h_cm / 100) ** 2

bmi = calc_bmi(employees[:, 0], employees[:, 1])
print('BMI per employee:', np.round(bmi, 1))
print('Average BMI     :', np.mean(bmi).round(1))

### 2.5 Indexing and slicing

For 2D arrays: `arr[row, col]`. Use `:` to select all rows or all columns.

In [ ]:
print(sales_reps[0])       # first rep — all features
print(sales_reps[0, 2])    # revenue of first rep
print(sales_reps[:, 1])    # deals_closed column for all reps
print(sales_reps[:3])      # first 3 reps

In [ ]:
# Select specific reps by index
sales_reps[[1, 3], :]

### 2.6 Boolean filtering

One of the most powerful tools in data analysis — select rows that satisfy a condition.

In [ ]:
calls   = sales_reps[:, 0]
revenue = sales_reps[:, 2]

# Boolean mask — one True/False per rep
high_revenue = revenue > 50
print('High-revenue reps mask:', high_revenue)

# Apply mask
print('Revenue of top performers:', revenue[high_revenue])

# Combine conditions: high revenue but not many calls (efficient reps)
efficient = (revenue > 50) & (calls < 110)
print('Efficient reps:')
print(sales_reps[efficient])

### 2.7 Sorting and ranking

`np.argsort` returns the *indices* that would sort an array — useful for top-N rankings.

In [ ]:
sort_idx = np.argsort(revenue)   # low to high

top3_idx = sort_idx[-3:][::-1]   # last 3, reversed = top 3
print('Top 3 reps by revenue:')
print(sales_reps[top3_idx])

### 2.8 Worked example — RFM customer scoring

**RFM analysis** is a battle-tested marketing technique that scores customers on three dimensions:
- **Recency (R)**: how recently they purchased (lower = better)
- **Frequency (F)**: how often they buy
- **Monetary (M)**: how much they spend

It's a good NumPy exercise because it combines vectorized arithmetic, filtering, and sorting.

In [ ]:
def generate_rfm_data(n=1000, seed=42):
    """Generate synthetic RFM data. Columns: recency (days), frequency, monetary (€)."""
    np.random.seed(seed)
    recency   = np.random.randint(1, 366, n).astype(float)
    frequency = np.random.randint(1, 26,  n).astype(float)
    monetary  = np.round(np.random.uniform(10, 10_000, n), 2)
    return np.column_stack([recency, frequency, monetary])

rfm = generate_rfm_data()
print('shape:', rfm.shape)
print('avg recency / frequency / monetary:', np.mean(rfm, axis=0).round(1))

In [ ]:
def rfm_score(data):
    """Higher score = better customer. Recency is inverted (recent = good)."""
    r, f, m = data[:, 0], data[:, 1], data[:, 2]
    return (np.max(r) - r) + f + (m / 1000)

scores   = rfm_score(rfm)
top5_idx = np.argsort(scores)[-5:][::-1]

print('Top 5 customers (recency, frequency, monetary):')
print(rfm[top5_idx].round(1))
print('Scores:', scores[top5_idx].round(1))

### ✏️ You try it — Airbnb price analysis

Below is a small array of Airbnb listings. Columns: `[price_€, num_reviews, availability_365]`.

1. What is the **mean and median price**? Why might they differ?
2. Filter to listings with **more than 30 reviews** (a proxy for demand). What is their average price?
3. Write a function `estimate_revenue(price, availability)` defined as `price × (365 − availability)` (days booked = 365 minus days listed as available). Apply it to all listings and find the **top 3 by estimated revenue**.

In [ ]:
# price_€, num_reviews, availability_365
listings = np.array([
    [ 85,  12, 300],
    [120,  48, 180],
    [200,   5, 365],
    [ 95,  31, 240],
    [340,   2,  90],
    [ 75, 110, 320],
    [180,  22, 150],
    [110,  67, 200],
])

# your code here

---
## 3. Preview: Your Airbnb Dataset

Download your city's `listings.csv` from [insideairbnb.com](https://insideairbnb.com/) and place it in the same folder as this notebook. We'll use Python's built-in `csv` module to peek at the structure before we have pandas available.

In [ ]:
import csv

with open('listings.csv', encoding='utf-8') as f:
    reader = csv.reader(f)
    headers = next(reader)
    rows = [next(reader) for _ in range(3)]

print(f'{len(headers)} columns. First few:')
for i, h in enumerate(headers[:15]):
    print(f'  {i:>3}: {h}')

In [ ]:
# Inspect the first row as a dictionary
first = dict(zip(headers, rows[0]))
for key, val in first.items():
    print(f'{key:35s}: {str(val)[:60]}')

**Things to notice:**
- `price` looks like `"$120.00"` — you need to strip the `$` and `,` before doing arithmetic
- Some columns are dates stored as strings, some values are empty (missing data)

In tomorrow's session we use **pandas** to handle all of this elegantly.

### ✏️ You try it — First look at the real data

Using only Python and NumPy (no pandas yet):

1. Load all values from the `price` column, clean the string format, and convert to a NumPy array of floats. Skip any rows where price is empty.
2. Print the mean, median, and 90th percentile price.
3. What share of listings costs more than €200/night?

In [ ]:
# Hint: use csv.DictReader for easy column access
# your code here

---
## 4. Using AI Tools Effectively

You are encouraged to use ChatGPT, Claude, or similar tools. The golden rule: **you must be able to explain every line**. Two prompting patterns that work well:

### Pattern 1 — Show your data, ask something specific

> *I have a NumPy array called `prices` with nightly Airbnb prices. Some values are 0 (likely errors). How do I replace them with `np.nan` and compute the mean ignoring those values?*

### Pattern 2 — Ask for an explanation, not just code

> *Explain what `np.percentile(arr, 75)` does and when I would use it in a pricing analysis.*

### What to watch out for
- AI tools sometimes import packages they don't use (e.g., importing `re` without ever calling it)
- They may use functions we haven't covered — that's fine, but **justify the choice** in your notebook
- Don't use AI for the **ideation phase** of the Airbnb assignment — the hypothesis should be yours

---
## What's next

Tomorrow we introduce **pandas**, which gives every column a name, handles mixed types, and makes the data cleaning we previewed today trivial. Everything you learned today — indexing, boolean filtering, summary statistics — carries directly over. You'll use it to load, clean, and explore the full Airbnb dataset for your assignment.

---
### Additional resources
- [NumPy quickstart](https://numpy.org/doc/stable/user/quickstart.html)
- [Inside Airbnb data dictionary](https://docs.google.com/spreadsheets/d/1iWCNJcSutYqpULSQHlNyGInUvHg2BoUGoNRIGa6Szc4)
- McKinney (2022) *Python for Data Analysis*, Ch. 4

### References
This notebook is based on an original version by Miguel Ángel Canela (2022) and Enric Junqué de Fortuny (2025).